In [1]:
# --- Бібліотеки для даної роботи ---
try:
    import numpy, pandas, matplotlib, plotly, sklearn, jupyterlab, ipywidgets
    print("Бібліотеки вже встановлені. Пропускаємо інсталяцію.")
except ImportError:
    print("Встановлюємо бібліотеки...")
    %pip install -q numpy pandas matplotlib plotly scikit-learn "jupyterlab>=3" "ipywidgets>=7.6"

Бібліотеки вже встановлені. Пропускаємо інсталяцію.


## Домашнє завдання: Тема 10. EM-алгоритм та розділення суміші Гаусівських функцій

### **Це допоможе закріпити такі навички:**

- Попередньої підготовки даних до моделювання
- Роботу з бібліотеками аналізу даних

### **Завдання (крок за кроком):**

***Для цієї задачі необхідно буде завантажити дані [World Happiness Report](https://www.kaggle.com/datasets/unsdsn/world-happiness).***

Для виконання завдання необхідно виконати такі кроки:

1. **Інсталювати та імпортувати необхідні бібліотеки:** 
    - Необхідно буде інсталювати такі пакети:
	```bash
	!pip install plotly==5.20.0
	!pip install "jupyterlab>=3" "ipywidgets>=7.6"
	```

2. **Завантажити дані:**
    - З набору https://www.kaggle.com/datasets/unsdsn/world-happiness.
	```bash
	!wget -O WorldHappinessReport.zip https://github.com/goitacademy/NUMERICAL-PROGRAMMING-IN-PYTHON/blob/main/WorldHappinessReport.zip?raw=true
	```

3. **Розпакувати дані:**
    ```bash
	!unzip WorldHappinessReport.zip
	```

4. **Прочитати дані та відобразити загальну інформацію про:**
	- Статистики
	- Типи ознак

5. **Побудувати діаграми розподілу числових ознак:**
    - Проаналізувати на відповідність чи не відповідність нормальному розподілу.

6. **Відібрати числових ознак та кореляційну матрицю:**
    - Виходячи із розуміння домену та даних відібрати певну кількість числових ознак
    - Відобразити кореляційну матрицю (*див. Тема 4. Вимірювання відстаней та подібностей в аналізі даних*)

7. **Зробити висновок про:**
    - Наявність та силу лінійного зв'язку між ознаками.

8. **Відобразити розподіл:**
    - Цільової ознаки (Happiness.Score або Happiness.Rank) за країнами.
    - Використовуючи наведений нижче код для побудови теплової мапи.
	```py
	fig = px.choropleth(data_dataframe,
						locations = "Country",
						color = "Happiness.Score",
						locationmode = "country names",
                    	)
	fig.update_layout(title = "Happiness Index 2017")
	fig.show()
	```

9. **Застосувати стандартизацію даних:**
    - Для приведення всіх значень до одного діапазону статистик.
    - Використовуючи функцію data_scale() та наступні перетворення
	```py
	def data_scale(data, scaler_type='minmax'):
	    from sklearn.preprocessing import MinMaxScaler
	    from sklearn.preprocessing import StandardScaler
	    from sklearn.preprocessing import Normalizer
	    if scaler_type == 'minmax':
	        scaler = MinMaxScaler()
	    if scaler_type == 'std':
	        scaler = StandardScaler()
	    if scaler_type == 'norm':
	        scaler = Normalizer()

	    scaler.fit(data)
	    res = scaler.transform(data)
	    return res

	data_scaled = data_scale(original_dataframe)
	df_scaled = pd.DataFrame(data_scaled, columns=[original_dataframe.columns])
	print(df_scaled.head())
	```

10. **Відобразити статистики:**
    - Отриманого стандартизованого набору даних та порівняти зі статистиками оригінального набору даних.
    - Зробити висновки.

11. **Побудувати модель кластеризації:**
	- Засобами функції `GaussianMixture()` бібліотеки `sklearn`.

12. **Побудувати теплову мапу:**
    - Для відображення розподілу країн за кластерами.

13. **Дослідити вплив:**
    - Різного набору ознак
    - Результат кластеризації

14. **Висновок:**
    - Зробити загальний висновок про відповідність результатів кластеризації оригінальному розподілу країн за ознакою.

**1. Імпорт необхідних бібліотек:**

In [45]:
# 1. СТАНДАРТНІ БІБЛІОТЕКИ PYTHON (Мережа, Файлова система, Попередження)
import os
import shutil
import urllib.request
import zipfile
import warnings

warnings.filterwarnings('ignore')

# 2. РОБОТА З ДАНИМИ ТА МАТЕМАТИКА
import math
import numpy as np
import pandas as pd

# 3. МАШИННЕ НАВЧАННЯ (Кластеризація та Препроцесинг)
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer

# 4. MLOps ТА СЕРІАЛІЗАЦІЯ МОДЕЛЕЙ
import joblib

# 5. ВІЗУАЛІЗАЦІЯ ТА UI (Plotly, IPywidgets & HTML)
from IPython.display import HTML, display
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats

print("📦 Модулі архітектури імпортовано успішно!")

📦 Модулі архітектури імпортовано успішно!


**1.3. Конфігурація експерименту (Глобальні змінні):**

In [41]:
# 1. МЕРЕЖА ТА ФАЙЛОВА СИСТЕМА
DATA_DIR                = "WorldHappinessDataSet"                               # Папка для ізольованого збереження всіх сирих даних
DATASET_URL             = "https://www.kaggle.com/api/v1/datasets/download/unsdsn/world-happiness"
ZIP_PATH                = os.path.join(DATA_DIR, "world-happiness.zip")

TARGET_YEAR             = "2017"                                                # Доступні роки: "2015" | "2016" | "2017" | "2018" | "2019"
CSV_FILENAME            = os.path.join(DATA_DIR, f"{TARGET_YEAR}.csv")          # Динамічний шлях до потрібного файлу

# 2. СТРУКТУРА ДАНИХ ТА ОЗНАКИ (Вирішення проблеми Schema Drift)
SCHEMA_MAPPING = {
    "2015": {
        "country": "Country",
        "target": "Happiness Score",
        "drop": ["Happiness Rank", "Standard Error", "Region"],
        "features": ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)", "Freedom", "Trust (Government Corruption)"]
    },
    "2016": {
        "country": "Country",
        "target": "Happiness Score",
        "drop": ["Happiness Rank", "Lower Confidence Interval", "Upper Confidence Interval", "Region"],
        "features": ["Economy (GDP per Capita)", "Family", "Health (Life Expectancy)", "Freedom", "Trust (Government Corruption)"]
    },
    "2017": {
        "country": "Country",
        "target": "Happiness.Score",
        "drop": ["Happiness.Rank", "Whisker.high", "Whisker.low"],
        "features": ["Economy..GDP.per.Capita.", "Family", "Health..Life.Expectancy.", "Freedom", "Trust..Government.Corruption."]
    },
    "2018": {
        "country": "Country or region",
        "target": "Score",
        "drop": ["Overall rank"],
        "features": ["GDP per capita", "Social support", "Healthy life expectancy", "Freedom to make life choices", "Perceptions of corruption"]
    },
    "2019": {
        "country": "Country or region",
        "target": "Score",
        "drop": ["Overall rank"],
        "features": ["GDP per capita", "Social support", "Healthy life expectancy", "Freedom to make life choices", "Perceptions of corruption"]
    }
}

CURRENT_SCHEMA          = SCHEMA_MAPPING[TARGET_YEAR]
COUNTRY_COL             = CURRENT_SCHEMA["country"]             # Динамічна колонка країни (змінювалась у 2018)
TARGET_METRIC           = CURRENT_SCHEMA["target"]              # Головна цільова метрика (Індекс щастя)
DROP_COLUMNS            = CURRENT_SCHEMA["drop"]                # Технічні колонки, що не несуть користі для кластеризації
FEATURES_FULL           = CURRENT_SCHEMA["features"]            # Повний набір соціально-економічних ознак для GMM
FEATURES_MINI           = [FEATURES_FULL[0], FEATURES_FULL[2]]  # Зменшений набір (ВВП та Здоров'я) для дослідження розмірності

# 3. МАШИННЕ НАВЧАННЯ (GMM) ТА СЕРІАЛІЗАЦІЯ
N_CLUSTERS              = 3                                     # Кількість кластерів: задає число прихованих Гаусівських розподілів (Високий, Середній, Низький рівень)
COVARIANCE_TYPE         = 'full'                                # Форма матриці коваріації (геометрія кластерів): 'full' - різні еліпси під будь-яким кутом | 'tied' - однакова форма та нахил для всіх | 'diag' - еліпси строго паралельні осям координат | 'spherical' - ідеальні круглі сфери різного радіусу
GMM_INIT_PARAMS         = 'kmeans'                              # Стратегія стартової ініціалізації (Крок 0 для EM): 'kmeans' - розумний розвідник для надійного старту | 'random' - повністю випадкові координати в просторі | 'random_from_data' - випадкові реальні точки з набору даних
SCALER_TYPE             = 'std'                                 # Алгоритм масштабування простору ознак: 'std' - центрує дисперсію навколо нуля (ідеально для GMM) | 'minmax' - жорстко стискає дані в межі від 0 до 1 | 'norm' - нормує самі вектори по їхній абсолютній довжині
N_INIT                  = 10                                    # Кількість перезапусків EM-алгоритму: захист від застрягання моделі в поганих локальних мінімумах
RANDOM_STATE            = 42                                    # Фіксація генератора псевдовипадкових чисел: гарантує 100% відтворюваність результатів експерименту

MODEL_DIR               = "GMM_Models"                          # Папка для збереження серіалізованих об'єктів
MODEL_PATH              = os.path.join(MODEL_DIR, f"gmm_{TARGET_YEAR}_{COVARIANCE_TYPE}_{GMM_INIT_PARAMS}_model.pkl") # Динамічне ім'я моделі
SCALER_PATH             = os.path.join(MODEL_DIR, f"scaler_{TARGET_YEAR}_{SCALER_TYPE}.pkl")                          # Динамічне ім'я скейлера

# 4. ВІЗУАЛІЗАЦІЯ ТА UI
PLOT_TEMPLATE           = "plotly_dark"                         # Темна тема для інтерактивних графіків Plotly
MAP_LOCATION_MODE       = "country names"                       # Режим розпізнавання країн для мап Choropleth
COLOR_SCALE_HAPPINESS   = "Viridis"                             # Безперервний градієнт для оригінального індексу щастя
COLOR_PALETTE_FULL      = px.colors.qualitative.Set1            # Контрастні дискретні кольори для 3-х кластерів (повний набір)
COLOR_PALETTE_MINI      = px.colors.qualitative.Pastel          # Пастельні кольори для експерименту зі зменшеною розмірністю

TABLE_PROPS             = {'background-color': '#1e1e1e', 'color': '#00c3ff', 'border': '1px solid #444', 'text-align': 'center'}
DESCRIBE_CMAP           = 'YlGn'                                # Кольорова схема (Yellow-Green) для підсвічування описових статистик

print(f"⚙️ Глобальні константи ініціалізовано!\n   Рік: {TARGET_YEAR} | GMM({COVARIANCE_TYPE}, {GMM_INIT_PARAMS}) + {SCALER_TYPE} Scaler")

⚙️ Глобальні константи ініціалізовано!
   Рік: 2017 | GMM(full, kmeans) + std Scaler


**1.7. Приклад на HTML (Анатомія GMM):**

In [10]:
C_RAW = "#888888"                               # Базовий колір для "нерозмічених" (сирих) даних у просторі
C_C1  = COLOR_PALETTE_FULL[0]                   # Динамічний колір Кластера 1 (підтягується з глобальної палітри констант)
C_C2  = COLOR_PALETTE_FULL[1]                   # Динамічний колір Кластера 2
C_C3  = COLOR_PALETTE_FULL[2]                   # Динамічний колір Кластера 3

html_em_pipeline = f"""
<div style="font-family: sans-serif; max-width: 900px; background-color: #111; padding: 20px; border-radius: 10px; border: 1px solid #333; margin: auto;">
    <h2 style="color: #00c3ff; text-align: center; margin-top: 0;">🧠 Анатомія GMM: Що робить EM-алгоритм з країнами?</h2>
    
    <div style="background-color: #1a1a1a; padding: 15px; margin-bottom: 15px; border-left: 5px solid {C_RAW}; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 0: Сирий простір (Дані після {SCALER_TYPE} Scaler)</div>
        <div style="color: {C_RAW}; font-size: 15px; margin-top: 5px; font-style: italic;">
            Маємо N країн у багатовимірному просторі ознак (ВВП, Здоров'я, Свобода...).<br>
            Усі точки "сірі", алгоритм ще нічого не знає про кластери.
        </div>
    </div>

    <div style="text-align: center; color: #ffd700; font-size: 20px;">⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #ffd700; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 1: Ініціалізація (Метод '{GMM_INIT_PARAMS}')</div>
        <div style="color: #ffd700; font-size: 15px; margin-top: 5px;">
            ШІ генерує {N_CLUSTERS} випадкові багатовимірні "дзвони" (Гаусівські розподіли).<br>
            Кожен має свій центр <b>(μ)</b> та матрицю коваріації <b>(Σ)</b>.
        </div>
    </div>

    <div style="text-align: center; color: #ff9900; font-size: 20px;">⬇ ♻️ Цикл EM-алгоритму ♻️ ⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #ff9900; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 2: E-крок (Expectation / Очікування)</div>
        <div style="color: #ff9900; font-size: 15px; margin-top: 5px;">
            Обчислення м'якої ймовірності (Soft Clustering) за формулою Баєса:<br>
            <i>"Країна Х належить до Кластера-1 на 10%, Кластера-2 на 85%, Кластера-3 на 5%".</i>
        </div>
    </div>

    <div style="text-align: center; color: #00aaff; font-size: 20px;">⬇</div>

    <div style="background-color: #1a1a1a; padding: 15px; margin: 15px 0; border-left: 5px solid #00aaff; border-radius: 5px;">
        <div style="color: #888; font-size: 12px; font-weight: bold; text-transform: uppercase;">Крок 3: M-крок (Maximization / Максимізація)</div>
        <div style="color: #00aaff; font-size: 15px; margin-top: 5px;">
            Оновлення параметрів дзвонів: <b>Нові μ</b> тягнуться до скупчень точок, <b>Нові Σ</b> змінюють форму еліпсів.
        </div>
    </div>

    <div style="text-align: center; color: #00ffcc; font-size: 20px; margin-top: 10px;">⬇</div>

    <div style="background-color: #222; padding: 15px; margin-top: 15px; border: 2px dashed #00ffcc; border-radius: 5px; text-align: center;">
        <div style="color: #888; font-size: 14px; font-weight: bold; text-transform: uppercase;">✓ Фінал: Збіжність (Convergence)</div>
        <div style="color: #00ffcc; font-size: 18px; margin-top: 10px; font-family: monospace;">[ <span style="color:{C_C1}">Кластер 1</span> | <span style="color:{C_C2}">Кластер 2</span> | <span style="color:{C_C3}">Кластер 3</span> ]</div>
    </div>
</div>
"""

print("Красивий Вивід (Інтерактивна схема логіки алгоритму):")
display(HTML(html_em_pipeline))

Красивий Вивід (Інтерактивна схема логіки алгоритму):


**2. Завантажити дані:**

In [36]:
def is_valid_zip(filepath):
    if not os.path.exists(filepath) or not zipfile.is_zipfile(filepath):
        return False
    try:
        with zipfile.ZipFile(filepath, 'r') as z:
            if z.testzip() is not None:
                return False
    except Exception:
        return False
    return True

def download_dataset():
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"⏳ Завантаження архіву у папку '{DATA_DIR}'...")
    try:
        urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)
        print("✅ Завантаження завершено.")
    except Exception as e:
        print(f"❌ Мережева помилка завантаження: {e}")

if os.path.exists(ZIP_PATH):
    print("🔍 Перевірка цілісності існуючого архіву...")
    if not is_valid_zip(ZIP_PATH):
        print("🪫 Архів пошкоджено. Видаляємо та завантажуємо наново...")
        os.remove(ZIP_PATH)
        download_dataset()
    else:
        print("🔋 Архів цілий. Пропускаємо мережевий запит.")
else:
    download_dataset()

⏳ Завантаження архіву у папку 'WorldHappinessDataSet'...
✅ Завантаження завершено.


**3. Розпакувати дані:**

In [43]:
if os.path.exists(CSV_FILENAME):
    print(f"⚡ Файл '{CSV_FILENAME}' вже розпаковано та готовий до роботи.")
elif os.path.exists(ZIP_PATH):
    if is_valid_zip(ZIP_PATH):
        print(f"📦 Аналізуємо вміст архіву '{ZIP_PATH}'...")
        try:
            with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
                csv_files = [f for f in zip_ref.namelist() if f.endswith('.csv')]
                if not csv_files:
                    raise Exception("В архіві немає CSV файлів!")

                expected_file_suffix = f"{TARGET_YEAR}.csv"
                target_csv_in_zip = next((f for f in csv_files if f.endswith(expected_file_suffix)), None)

                if not target_csv_in_zip:
                    print(f"   ⚠️ Доступні файли в архіві: {csv_files}")
                    raise Exception(f"Файл для {TARGET_YEAR} року не знайдено в архіві!")

                print(f"   🎯 Знайдено цільовий файл: '{target_csv_in_zip}'")

                tmp_csv_path = CSV_FILENAME + ".tmp"

                try:
                    print(f"   ⚙️ Витягуємо '{target_csv_in_zip}' атомарно...")
                    with zip_ref.open(target_csv_in_zip) as source, open(tmp_csv_path, "wb") as target:
                        shutil.copyfileobj(source, target)

                    if os.path.exists(CSV_FILENAME):
                        os.remove(CSV_FILENAME)
                    os.rename(tmp_csv_path, CSV_FILENAME)
                    print(f"✅ Успіх! Файл '{CSV_FILENAME}' (дані {TARGET_YEAR} року) збережено безпечно.")

                except PermissionError:
                    raise Exception(f"Файл {CSV_FILENAME} заблоковано іншою програмою. Закрийте Excel або інші скрипти.")
                except Exception as extract_err:
                    raise Exception(f"Помилка фізичного запису на диск: {extract_err}")
                finally:
                    if os.path.exists(tmp_csv_path):
                        os.remove(tmp_csv_path)

        except Exception as e:
            print(f"❌ Системна помилка під час роботи з архівом: {e}")
    else:
        print("❌ Критична помилка: Архів досі пошкоджений.")
else:
    print("❌ Помилка: Архів не знайдено. Перезапустіть попередній блок завантаження.")

📦 Аналізуємо вміст архіву 'WorldHappinessDataSet/world-happiness.zip'...
   🎯 Знайдено цільовий файл: '2017.csv'
   ⚙️ Витягуємо '2017.csv' атомарно...
✅ Успіх! Файл 'WorldHappinessDataSet/2017.csv' (дані 2017 року) збережено безпечно.


**4. Прочитати дані та відобразити загальну інформацію:**

In [59]:
print(f"📂 Завантаження набору даних з файлу: {CSV_FILENAME}\n")
df = pd.read_csv(CSV_FILENAME)

print("Красивий Вивід - Перші 5 рядків набору даних:")
display(df.head().style.background_gradient(cmap='Blues').set_properties(**TABLE_PROPS))

print("Технічний Вивід:\nОписові статистики:")
display(df.describe().T.style.background_gradient(cmap=DESCRIBE_CMAP).format("{:.4f}"))

print("Інформація про типи ознак та пропуски:")
df.info()

📂 Завантаження набору даних з файлу: WorldHappinessDataSet/2017.csv

Красивий Вивід - Перші 5 рядків набору даних:


,Country,Happiness.Rank,Happiness.Score,Whisker.high,Whisker.low,Economy..GDP.per.Capita.,Family,Health..Life.Expectancy.,Freedom,Generosity,Trust..Government.Corruption.,Dystopia.Residual
0,Norway,1,7.537000,7.594445,7.479556,1.616463,1.533524,0.796667,0.635423,0.362012,0.315964,2.277027
1,Denmark,2,7.522000,7.581728,7.462272,1.482383,1.551122,0.792566,0.626007,0.355280,0.400770,2.313707
2,Iceland,3,7.504000,7.622030,7.385970,1.480633,1.610574,0.833552,0.627163,0.475540,0.153527,2.322715
3,Switzerland,4,7.494000,7.561772,7.426227,1.564980,1.516912,0.858131,0.620071,0.290549,0.367007,2.276716
4,Finland,5,7.469000,7.527542,7.410458,1.443572,1.540247,0.809158,0.617951,0.245483,0.382612,2.430182


Технічний Вивід:
Описові статистики:


,count,mean,std,min,25%,50%,75%,max
Happiness.Rank,155.0000,78.0000,44.8888,1.0000,39.5000,78.0000,116.5000,155.0000
Happiness.Score,155.0000,5.3540,1.1312,2.6930,4.5055,5.2790,6.1015,7.5370
Whisker.high,155.0000,5.4523,1.1185,2.8649,4.6082,5.3700,6.1946,7.6220
Whisker.low,155.0000,5.2557,1.1450,2.5211,4.3750,5.1932,6.0065,7.4796
Economy..GDP.per.Capita.,155.0000,0.9847,0.4208,0.0000,0.6634,1.0646,1.3180,1.8708
Family,155.0000,1.1889,0.2873,0.0000,1.0426,1.2539,1.4143,1.6106
Health..Life.Expectancy.,155.0000,0.5513,0.2371,0.0000,0.3699,0.6060,0.7230,0.9495
Freedom,155.0000,0.4088,0.1500,0.0000,0.3037,0.4375,0.5166,0.6582
Generosity,155.0000,0.2469,0.1348,0.0000,0.1541,0.2315,0.3238,0.8381
Trust..Government.Corruption.,155.0000,0.1231,0.1017,0.0000,0.0573,0.0898,0.1533,0.4643


Інформація про типи ознак та пропуски:
<class 'pandas.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        155 non-null    str    
 1   Happiness.Rank                 155 non-null    int64  
 2   Happiness.Score                155 non-null    float64
 3   Whisker.high                   155 non-null    float64
 4   Whisker.low                    155 non-null    float64
 5   Economy..GDP.per.Capita.       155 non-null    float64
 6   Family                         155 non-null    float64
 7   Health..Life.Expectancy.       155 non-null    float64
 8   Freedom                        155 non-null    float64
 9   Generosity                     155 non-null    float64
 10  Trust..Government.Corruption.  155 non-null    float64
 11  Dystopia.Residual              155 non-null    float64
dtypes: float64(10), int64(

**5. Побудувати діаграми розподілу числових ознак:**

In [64]:
raw_features = [TARGET_METRIC] + FEATURES_FULL
features_to_plot = [f for f in raw_features if f in df.columns]

num_plots = len(features_to_plot)
cols = 3
rows = math.ceil(num_plots / cols)

clean_titles = [f"<b>{str(f).replace('.', ' ').replace('_', ' ').strip()}</b>" for f in features_to_plot]

fig_dist = make_subplots(
    rows=rows, cols=cols, 
    subplot_titles=clean_titles,
    vertical_spacing=0.18,
    horizontal_spacing=0.08
)

for i, feature in enumerate(features_to_plot):
    row = (i // cols) + 1
    col = (i % cols) + 1
    data = df[feature].dropna()

    fig_dist.add_trace(
        go.Histogram(
            x=data, histnorm='probability density', 
            name=f"{feature}", marker_color='#00c3ff', opacity=0.6, nbinsx=25,
            hovertemplate="<b>Діапазон значень:</b> %{x}<br><b>Емпірична щільність:</b> %{y:.4f}<extra></extra>"
        ), row=row, col=col
    )

    mu, std = data.mean(), data.std()
    x_curve = np.linspace(data.min(), data.max(), 100)
    y_curve = stats.norm.pdf(x_curve, mu, std)

    fig_dist.add_trace(
        go.Scatter(
            x=x_curve, y=y_curve, mode='lines', 
            name=f"Ідеальний Гаусс", line=dict(color='#ffd700', width=3, dash='dot'),
            hovertemplate="<b>Ідеальний Гаусс</b><br>Значення ознаки: %{x:.2f}<br>Теоретична щільність: %{y:.4f}<extra></extra>"
        ), row=row, col=col
    )

    fig_dist.update_xaxes(title_text="Значення", title_font=dict(size=11, color="#888"), showgrid=True, gridcolor='#333', row=row, col=col)
    fig_dist.update_yaxes(title_text="Щільність", title_font=dict(size=11, color="#888"), showgrid=True, gridcolor='#333', row=row, col=col)

fig_dist.update_layout(
    height=350 * rows + 80, 
    width=1200, 
    title_text="📊 Перевірка на нормальність: Реальний розподіл vs Ідеальний Дзвін Гаусса", 
    title_x=0.5,
    template=PLOT_TEMPLATE,
    showlegend=False,
    hovermode="x unified",
    margin=dict(b=100),
    annotations=[
        dict(
            x=0.5, y=-0.075, xref="paper", yref="paper",
            text="🟡 <b>Жовтий пунктир</b> — ідеальна математична модель (Дзвін Гаусса).<br>🟦 <b>Блакитні стовпці</b> — реальний емпіричний розподіл ознаки в датасеті.",
            showarrow=False, font=dict(size=14, color="#cccccc"), align="center", xanchor="center", yanchor="top"
        )
    ]
)

print("Красивий Вивід:")
fig_dist.show()

Красивий Вивід:


**5.1. Висновок до Кроку 5:**

### Аналіз нормальності розподілу

Для кожної ознаки ми побудували ідеальну теоретичну криву (жовтий пунктир), яка описується рівнянням щільності одномірного нормального розподілу:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}$$

де $\mu$ — математичне сподівання (середнє значення), а $\sigma$ — стандартне відхилення ознаки. Порівнюючи емпіричні гістограми з цією теоретичною моделлю, робимо такі висновки щодо природи соціально-економічних даних:

1. **Відсутність ідеальної нормальності:** Більшість ознак у датасеті **не мають** ідеально симетричного нормального розподілу. Реальні макроекономічні дані схильні до перекосів (Skewness) та нетипових викидів.
2. **Лівостороння асиметрія (Negative Skew):** Ознаки `Economy (GDP)` та `Health (Life Expectancy)` помітно зміщені вправо. Це означає, що у світі переважають країни із середнім та високим рівнем життя, тоді як країни з абсолютною бідністю формують довгий, але тонкий лівий "хвіст".
3. **Правостороння асиметрія (Positive Skew):** Ознака `Trust (Government Corruption)` має яскраво виражений експоненційний спад. Переважна більшість країн має дуже низький індекс довіри до уряду, і лише одиничні геополітичні "аномалії" (як-от країни Скандинавії) мають високі показники, утворюючи правий "хвіст".
4. **Вплив на GMM-кластеризацію:** Жорсткий алгоритм K-Means працює погано з такими асиметричними даними, оскільки намагається вписати точки в ідеальні сфери. Натомість `GaussianMixture` (Модель суміші Гаусів) чудово впорається з цим завданням з двох причин:
    - Використання повної матриці коваріації (`covariance_type='full'`) дозволяє кластерам набувати форми витягнутих еліпсів.
    - Математично доведено, що лінійна комбінація кількох Гаусівських розподілів здатна апроксимувати будь-який, навіть найскладніший і найбільш асиметричний розподіл даних.

**6. Відібрати числових ознак та кореляційну матрицю:**

In [67]:
print("🔍 Відбір числових ознак для аналізу...")

selected_columns = [TARGET_METRIC] + [f for f in FEATURES_FULL if f in df.columns]
df_selected = df[selected_columns].copy()

df_numeric = df_selected.select_dtypes(include=[np.number])

print(f"✅ Відібрано числових ознак: {len(df_numeric.columns)} (включно з цільовою метрикою).")
print("📊 Побудова матриці кореляцій...")

corr_matrix = df_numeric.corr()

clean_labels = [str(col).replace('.', ' ').replace('_', ' ').strip() for col in corr_matrix.columns]

fig_corr = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=clean_labels,
    y=clean_labels,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=np.round(corr_matrix.values, 2),
    texttemplate="%{text}",
    textfont={"size": 13, "color": "white"},
    hoverinfo="text",
    hovertemplate="<b>Ознака X:</b> %{x}<br><b>Ознака Y:</b> %{y}<br><b>Кореляція:</b> %{z:.4f}<extra></extra>"
))

fig_corr.update_layout(
    title_text="🔥 Матриця кореляцій Пірсона (Взаємозв'язок факторів)",
    title_x=0.5,
    width=750,
    height=750,
    template=PLOT_TEMPLATE,
    xaxis=dict(tickangle=-45),
    margin=dict(b=120)
)

print("\nКрасивий Вивід:")
fig_corr.show()

🔍 Відбір числових ознак для аналізу...
✅ Відібрано числових ознак: 6 (включно з цільовою метрикою).
📊 Побудова матриці кореляцій...

Красивий Вивід:


**7. Зробити висновок:**

### Аналіз наявності та сили лінійного зв'язку

Для оцінки взаємозв'язків ми побудували теплову карту на основі **коефіцієнта кореляції Пірсона ($r$)**, який обчислюється за формулою:

$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$$

де $\bar{x}$ та $\bar{y}$ — середні значення відповідних ознак. Цей коефіцієнт варіюється в межах $[-1; 1]$, де $1$ — ідеальна пряма залежність, $0$ — відсутність лінійного зв'язку, а $-1$ — ідеальна зворотна залежність. 

Аналізуючи отриману матрицю, можна зробити такі висновки щодо відібраних числових ознак:

1. **Сильний позитивний зв'язок із цільовою метрикою:** Найбільший вплив на Індекс щастя (`Happiness Score`) мають економічний фактор `Economy (GDP per Capita)` та соціально-медичний фактор `Health (Life Expectancy)`. Коефіцієнт кореляції для них перевищує 0.75, що вказує на виражену пряму лінійну залежність.
2. **Внутрішня мультиколінеарність (Зв'язок між ознаками):** Спостерігається дуже сильний взаємозв'язок (кореляція ~0.8) між самим ВВП та тривалістю життя. Це логічно з точки зору домену (багатші країни мають кращу медицину). 
3. **Слабкі та помірні зв'язки:** Ознака `Trust (Government Corruption)` демонструє значно слабший зв'язок як з індексом щастя, так і з іншими метриками (кореляція в діапазоні 0.2 - 0.4). Це свідчить про те, що цей фактор є більш незалежним і додає системі унікальної дисперсії.
4. **Вплив на подальшу кластеризацію:** Наявність сильної мультиколінеарності означає, що хмара даних у багатовимірному просторі має форму витягнутого еліпсоїда, а не ідеальної сфери. Саме тому використання алгоритму Gaussian Mixture Model (GMM) з параметром `covariance_type='full'` є архітектурно правильним рішенням — цей алгоритм здатен адаптуватися до таких лінійних зв'язків та коректно розділити простір.

**8. Відобразити розподіл:**

In [76]:
print(f"🌍 Побудова теплової мапи для цільової ознаки '{TARGET_METRIC}'...\n")

fig_original_map = px.choropleth(
    df,
    locations=COUNTRY_COL,             
    color=TARGET_METRIC,               
    locationmode=MAP_LOCATION_MODE,    
    color_continuous_scale=COLOR_SCALE_HAPPINESS, 
    hover_name=COUNTRY_COL
)

fig_original_map.update_layout(
    title_text=f"🗺️ Happiness Index {TARGET_YEAR} (Оригінальні дані)",
    title_x=0.5,
    template=PLOT_TEMPLATE,            
    geo=dict(
        showframe=False,
        showcoastlines=True, coastlinecolor="rgba(255, 255, 255, 0.2)",
        projection_type='natural earth',
        bgcolor='rgba(0,0,0,0)',
        lakecolor='#111111',
        landcolor='#222222'
    ),
    width=1200, height=700
)

print("Красивий Вивід - Оригінальна мапа щастя:")
fig_original_map.show()

print("Технічний Вивід - Екстремуми рейтингу (ТОП-5 та Анти-ТОП-5 країн):")
top_bottom_df = pd.concat([
    df[[COUNTRY_COL, TARGET_METRIC]].nlargest(5, TARGET_METRIC),
    df[[COUNTRY_COL, TARGET_METRIC]].nsmallest(5, TARGET_METRIC).sort_values(by=TARGET_METRIC, ascending=False)
])

display(top_bottom_df.style.background_gradient(cmap='YlGnBu', subset=[TARGET_METRIC])\
        .set_properties(**TABLE_PROPS).format({TARGET_METRIC: "{:.4f}"}))

🌍 Побудова теплової мапи для цільової ознаки 'Happiness.Score'...

Красивий Вивід - Оригінальна мапа щастя:


Технічний Вивід - Екстремуми рейтингу (ТОП-5 та Анти-ТОП-5 країн):


,Country,Happiness.Score
0,Norway,7.5370
1,Denmark,7.5220
2,Iceland,7.5040
3,Switzerland,7.4940
4,Finland,7.4690
150,Rwanda,3.4710
151,Syria,3.4620
152,Tanzania,3.3490
153,Burundi,2.9050
154,Central African Republic,2.6930


**9. Застосувати стандартизацію даних:**

In [ ]:
print(f"⚖️ Масштабування ознак через функцію data_scale() (Метод: {SCALER_TYPE.upper()})...")

def data_scale(data, scaler_type='minmax'):
    if scaler_type == 'minmax':
        scaler = MinMaxScaler()
    elif scaler_type == 'std':
        scaler = StandardScaler()
    elif scaler_type == 'norm':
        scaler = Normalizer()
    else:
        raise ValueError("Невідомий тип скейлера!")

    scaler.fit(data)
    res = scaler.transform(data)

    return res, scaler

X_features = df[FEATURES_FULL].copy()
data_scaled, fitted_scaler = data_scale(X_features, scaler_type=SCALER_TYPE)
df_scaled = pd.DataFrame(data_scaled, columns=FEATURES_FULL, index=df.index)

print("✅ Масштабування завершено успішно! (Скейлер збережено в оперативній пам'яті)\n")

f1, f2 = FEATURES_FULL[0], FEATURES_FULL[1]
f1_clean = str(f1).replace('.', ' ').strip()
f2_clean = str(f2).replace('.', ' ').strip()

fig_scale = make_subplots(
    rows=1, cols=2, 
    subplot_titles=(f"Оригінальний масштаб", f"Відмасштабовано ({SCALER_TYPE})"),
    horizontal_spacing=0.1
)

fig_scale.add_trace(go.Scatter(
    x=df[f1], y=df[f2], mode='markers',
    marker=dict(color='#00c3ff', size=9, opacity=0.7, line=dict(width=1, color='black')),
    text=df[COUNTRY_COL], 
    hovertemplate="<b>%{text}</b><br>" + f"{f1_clean}: %{{x:.3f}}<br>{f2_clean}: %{{y:.3f}}<extra></extra>"
), row=1, col=1)

fig_scale.add_trace(go.Scatter(
    x=df_scaled[f1], y=df_scaled[f2], mode='markers',
    marker=dict(color='#ccff00', size=9, opacity=0.7, line=dict(width=1, color='black')),
    text=df[COUNTRY_COL], 
    hovertemplate="<b>%{text}</b><br>Scaled X: %{x:.3f}<br>Scaled Y: %{y:.3f}<extra></extra>"
), row=1, col=2)

fig_scale.update_xaxes(title_text=f1_clean, showgrid=True, gridcolor='#333', row=1, col=1)
fig_scale.update_yaxes(title_text=f2_clean, showgrid=True, gridcolor='#333', row=1, col=1)

fig_scale.update_xaxes(title_text="Scaled X", showgrid=True, gridcolor='#333', row=1, col=2)
fig_scale.update_yaxes(title_text="Scaled Y", showgrid=True, gridcolor='#333', row=1, col=2)

fig_scale.update_layout(
    title_text="✨ Трансформація простору: Оригінальні vs Відмасштабовані дані",
    title_x=0.5, width=1100, height=500, template=PLOT_TEMPLATE, showlegend=False,
    margin=dict(b=80),
    annotations=[
        dict(
            x=0.5, y=-0.20, xref="paper", yref="paper",
            text="💡 Зверніть увагу на осі: форма хмари точок зберігається, але масштаб змінюється відповідно до обраного алгоритму.",
            showarrow=False, font=dict(size=14, color="#cccccc"), align="center"
        )
    ]
)

print("Красивий Вивід - Візуалізація ефекту масштабування:")
fig_scale.show()

print("Технічний Вивід - Перші 5 рядків відмасштабованих ознак:")
display(df_scaled.head().style.background_gradient(cmap='Purples').set_properties(**TABLE_PROPS).format("{:.4f}"))

⚖️ Масштабування ознак через функцію data_scale() (Метод: STD)...
✅ Масштабування завершено успішно! (Скейлер збережено в оперативній пам'яті)

Красивий Вивід - Візуалізація ефекту масштабування:


Технічний Вивід (Перші 5 рядків відмасштабованих ознак):


,Economy..GDP.per.Capita.,Family,Health..Life.Expectancy.,Freedom,Trust..Government.Corruption.
0,1.5062,1.2036,1.0382,1.5158,1.9031
1,1.1865,1.2650,1.0208,1.4529,2.7400
2,1.1823,1.4727,1.1943,1.4606,0.3001
3,1.3834,1.1456,1.2983,1.4132,2.4068
4,1.0940,1.2271,1.0910,1.3990,2.5608


**10. Відобразити статистики:**

In [96]:
print("📊 Технічний аудит та порівняння описових статистик...\n")

fig_stats = make_subplots(
    rows=2, cols=1,
    subplot_titles=("1. Розподіл ОРИГІНАЛЬНИХ ознак (Різні масштаби та дисперсії)", f"2. Розподіл ВІДМАСШТАБОВАНИХ ознак ({SCALER_TYPE.upper()})"),
    vertical_spacing=0.12
)

colors = px.colors.qualitative.Pastel

for i, col in enumerate(FEATURES_FULL):
    clean_name = str(col).replace('.', ' ').strip()
    fig_stats.add_trace(go.Violin(
        x=X_features[col], name=clean_name, marker_color=colors[i % len(colors)],
        box_visible=True,
        meanline_visible=True,
        points='outliers',
        hovertemplate="<b>%{name}</b><br>Значення: %{x}<extra></extra>"
    ), row=1, col=1)

for i, col in enumerate(FEATURES_FULL):
    clean_name = str(col).replace('.', ' ').strip()
    fig_stats.add_trace(go.Violin(
        x=df_scaled[col], name=clean_name, marker_color=colors[i % len(colors)],
        box_visible=True,
        meanline_visible=True,
        points='outliers',
        hovertemplate="<b>%{name} (Scaled)</b><br>Значення: %{x}<extra></extra>"
    ), row=2, col=1)

fig_stats.update_xaxes(title_text="Оригінальні значення", showgrid=True, gridcolor='#333', row=1, col=1)
fig_stats.update_yaxes(title_text="Ознаки", showgrid=True, gridcolor='#333', row=1, col=1)

fig_stats.update_xaxes(title_text="Відмасштабовані значення", showgrid=True, gridcolor='#333', row=2, col=1)
fig_stats.update_yaxes(title_text="Ознаки", showgrid=True, gridcolor='#333', row=2, col=1)

fig_stats.update_layout(
    title_text="🎻 'Скрипкові діаграми' (Violin Plots): Аналіз щільності та квартилів",
    title_x=0.5, 
    width=1500, height=950,
    template=PLOT_TEMPLATE, showlegend=False,
    margin=dict(b=130, l=150, t=80),
    annotations=[
        dict(
            x=0.5, y=-0.13, xref="paper", yref="paper",
            text="💡 <b>Violin Plot</b> поєднує Boxplot (всередині) та хвилю щільності (зовні). Товщина 'скрипки' показує, де сконцентровано найбільше країн.<br>Зверніть увагу, як нижній графік вирівняв дисперсію всіх ознак, підготувавши їх до GMM!",
            showarrow=False, font=dict(size=14, color="#cccccc"), align="center"
        )
    ]
)

print("Красивий Вивід:")
fig_stats.show()

print("Технічний Вивід 1 - Статистики ОРИГІНАЛЬНОГО набору (Для порівняння):")
display(X_features.describe().T.style.background_gradient(cmap=DESCRIBE_CMAP).format("{:.4f}"))

print("\nТехнічний Вивід 2 - Статистики ВІДМАСШТАБОВАНОГО набору (Scaled):")
display(df_scaled.describe().T.style.background_gradient(cmap='Purples').format("{:.4f}"))

📊 Технічний аудит та порівняння описових статистик...

Красивий Вивід:


Технічний Вивід 1 - Статистики ОРИГІНАЛЬНОГО набору (Для порівняння):


,count,mean,std,min,25%,50%,75%,max
Economy..GDP.per.Capita.,155.0000,0.9847,0.4208,0.0000,0.6634,1.0646,1.3180,1.8708
Family,155.0000,1.1889,0.2873,0.0000,1.0426,1.2539,1.4143,1.6106
Health..Life.Expectancy.,155.0000,0.5513,0.2371,0.0000,0.3699,0.6060,0.7230,0.9495
Freedom,155.0000,0.4088,0.1500,0.0000,0.3037,0.4375,0.5166,0.6582
Trust..Government.Corruption.,155.0000,0.1231,0.1017,0.0000,0.0573,0.0898,0.1533,0.4643



Технічний Вивід 2 - Статистики ВІДМАСШТАБОВАНОГО набору (Scaled):


,count,mean,std,min,25%,50%,75%,max
Economy..GDP.per.Capita.,155.0000,-0.0000,1.0032,-2.3477,-0.7661,0.1904,0.7947,2.1125
Family,155.0000,0.0000,1.0032,-4.1521,-0.5108,0.2271,0.7873,1.4727
Health..Life.Expectancy.,155.0000,-0.0000,1.0032,-2.3332,-0.7680,0.2315,0.7265,1.6849
Freedom,155.0000,0.0000,1.0032,-2.7341,-0.7030,0.1917,0.7208,1.6685
Trust..Government.Corruption.,155.0000,-0.0000,1.0032,-1.2150,-0.6498,-0.3284,0.2978,3.3670


**10.1. Висновок до Кроку 10:**

### Аналіз стандартизованих статистик та щільності розподілу

Порівнюючи описові статистики (`describe()`) та їх візуалізацію через Скрипкові діаграми (Violin Plots) для оригінального та відмасштабованого наборів даних, можна зробити такі математичні висновки:

1. **Трансформація простору (Feature Scaling):** В оригінальному наборі ознаки мали кардинально різні математичні діапазони. Після застосування нашого скейлера всі вектори ознак $x$ були лінійно трансформовані у новий простір $x'$. Наприклад, при стандартизації (StandardScaler) це досягається зміщенням середнього до нуля та нормуванням дисперсії до одиниці:
    $$x'_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$
    Це жорстко вписало всі ознаки в єдиний стандартизований масштаб, що яскраво видно по вирівняних "скрипках" на нижньому графіку. 

2. **Уніфікація статистичної ваги:** Завдяки масштабуванню не лише медіани, але й дисперсії (ширина хвилі розподілу) були збалансовані. Це означає, що відтепер жодна ознака не зможе штучно "домінувати" в алгоритмі просто за рахунок більших абсолютних чисел (наприклад, макроекономіка більше не задавить соціальні фактори).

3. **Математична стабільність для GMM:** Скрипкові діаграми наочно показують оцінку щільності (Kernel Density). Наявність "потовщень" вказує на скупчення країн. Алгоритм GMM шукає такі згущення, щоб описати їх багатовимірним нормальним розподілом:
    $$\mathcal{N}(x | \mu, \Sigma) = \frac{1}{\sqrt{(2\pi)^D |\Sigma|}} \exp\left(-\frac{1}{2}(x - \mu)^T \Sigma^{-1} (x - \mu)\right)$$
    Де $\Sigma$ — матриця коваріації, а $D$ — розмірність простору. Якби ми не відмасштабували дані, детермінант $|\Sigma|$ та обернена матриця $\Sigma^{-1}$ були б екстремально спотворені ознакою з найбільшою дисперсією. Масштабування гарантує, що багатовимірні еліпси Гаусса формуватимуться на основі реальної геометрії даних, а не через різницю в одиницях виміру.

**11. Побудувати модель кластеризації:**

**12. Побудувати теплову мапу:**

**13. Дослідити вплив:**

**13.5.\*\* Експорт навченої моделі ШІ:**

**13.9.\*\* Класифікація щастя:**

**14. Висновок:**